# 03 Feature selection

Purpose: choose the features for the scorecard and the challengers.
All screening uses the training sample. Out-of-time is used only for stability checks.

In [ ]:
import sys, importlib
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import src.feature_selection as fs
importlib.reload(fs)  # always load the current module from disk
assert "Heavy zero mass" in fs._bin_numeric.__doc__, "old feature_selection.py is loaded; restart the kernel"

from src import config
from src.splits import assign_sample
from src.data_prep import label_summary
from src.eda import feature_psi, missingness_by_year
from src.feature_selection import (
    CANDIDATE_FEATURES, CATEGORICAL, add_derived_features, assert_no_leakage,
    correlated_pairs, iv_screen, information_value, vif, woe_table,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

raw = assign_sample(pd.read_parquet((lambda p: p if p.exists() else config.PROCESSED_PATH)(getattr(config, "SLIM_PATH", config.PROCESSED_PATH.with_name("loans_slim.parquet")))))
df = add_derived_features(raw[raw["sample"] != "excluded"].copy())
print(f"Modelling loans: {len(df):,}")
print("module loaded from:", fs.__file__)

## 1. Leakage check

In [ ]:
assert_no_leakage(CANDIDATE_FEATURES)
print("Leakage check passed. Candidates:", len(CANDIDATE_FEATURES))

## 2. Information Value screen (training sample)

Bands: below 0.02 drop, 0.02 to 0.10 weak, 0.10 to 0.30 medium, above 0.30 strong,
above 0.50 check for leakage. Screening guides, not hard rules.

In [ ]:
iv = iv_screen(df, CANDIDATE_FEATURES, bins=5)
display(iv.round(4))

# guard: these features must have signal after the binning fix
check = iv.set_index("feature")["iv"]
assert check["term_months"] > 0, "term_months IV is still zero: old binning code is loaded"
print("guard passed: term_months IV =", round(check["term_months"], 4))

## 3. Stability: PSI and missingness, training against out-of-time

In [ ]:
train_s = df[df["sample"] == "train"]
oot_s = df[df["sample"] == "oot"]

numeric_candidates = [f for f in CANDIDATE_FEATURES if f not in CATEGORICAL]
psi_derived = feature_psi(train_s, oot_s, numeric_candidates)
display(psi_derived.round(4))

miss = pd.DataFrame({
    "train_missing": train_s[CANDIDATE_FEATURES].isna().mean(),
    "oot_missing": oot_s[CANDIDATE_FEATURES].isna().mean(),
})
miss["change"] = miss["oot_missing"] - miss["train_missing"]
display(miss.round(4).sort_values("change", key=abs, ascending=False))

Flag any feature whose missing share moves by more than about 5 points.

## 4. revol_util by year and grade mix by sample

In [ ]:
year = df["issue_date"].dt.year
util = pd.DataFrame({
    "median": df.groupby(year)["revol_util"].median(),
    "share_at_0": df.groupby(year)["revol_util"].apply(lambda s: (s == 0).mean()),
    "share_at_100plus": df.groupby(year)["revol_util"].apply(lambda s: (s >= 100).mean()),
})
display(util.round(4))
display(pd.crosstab(df["sample"], df["grade"], normalize="index").round(3))

A step change in `revol_util` at one date points to a definition change.
A gradual move points to borrower behaviour. A stable grade mix with rising
defaults points to concept drift within grades.

## 5. Correlation and VIF on the shortlist (IV at least 0.02)

In [ ]:
SHORTLIST_IV = 0.02
shortlist = iv.loc[iv["iv"] >= SHORTLIST_IV, "feature"].tolist()
numeric_short = [f for f in shortlist if f not in CATEGORICAL]
print(f"Shortlist: {len(shortlist)} features")
print(shortlist)

pairs = correlated_pairs(train_s, numeric_short, threshold=0.7)
display(pairs if len(pairs) else pd.DataFrame({"result": ["no pairs above |r| = 0.7"]}))
display(vif(train_s, numeric_short).round(2).to_frame("vif"))

Rules: from each pair with |r| above 0.7, drop the feature with lower IV unless
it is easier to explain. Drop any feature with VIF above 10 after that.

## 6. Decision table

In [ ]:
decision = iv[["feature", "iv", "band"]].copy()
decision["in_shortlist"] = decision["feature"].isin(shortlist)
decision["psi_train_oot"] = decision["feature"].map(psi_derived.set_index("feature")["psi"])
decision["missing_change"] = decision["feature"].map(miss["change"])
decision["decision"] = ""
display(decision.round(4))

Record keep, drop or review with the reason in `docs/feature_decisions.md`.

## 7. Derogatory counts: share of training loans with a non-zero value


In [ ]:
DEROG = ["delinq_2yrs", "pub_rec", "has_delinquency", "collections_12_mths_ex_med"]
nonzero = pd.DataFrame({
    "train_nonzero_share": train_s[DEROG].gt(0).mean(),
    "train_loans_nonzero": train_s[DEROG].gt(0).sum(),
    "train_bad_rate_if_nonzero": [train_s.loc[train_s[c] > 0, "target"].mean() for c in DEROG],
    "train_bad_rate_if_zero": [train_s.loc[train_s[c] == 0, "target"].mean() for c in DEROG],
})
display(nonzero.round(4))
